## 01 - Prédiction de l'humidité du sol à J+1

Dans ce notebook, nous allons construire un modèle de régression pour prédire l'humidité du sol au jour suivant.

La variable cible est `Soil_Moisture_J1`.

Le modèle utilisé est `RandomForestRegressor`.

L'évaluation sera faite avec :

- MAE
- RMSE
- R²



In [3]:

import pandas as pd
import numpy as np


In [5]:
df = pd.read_csv(r"C:\Users\kerro\OneDrive\Bureau\smart-agri\data\processed\irrigation_prediction_cols19.csv")

df.head(15)

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need,Stress_Index,Soil_Moisture_J1
0,Clay,6.14,36.48,0.42,2.17,21.90,31.19,1167.70,4.01,1.97,Wheat,Vegetative,Rabi,Yes,1.98,South,Low,38.53,42.73500
1,Silt,6.41,50.56,0.38,0.23,36.50,26.01,831.28,10.72,16.82,Maize,Flowering,Zaid,Yes,33.56,Central,Medium,47.40,43.26370
2,Sandy,7.71,40.07,1.09,2.18,41.83,76.41,1844.45,7.75,19.03,Cotton,Harvest,Rabi,Yes,34.62,South,Low,42.22,50.22325
3,Clay,5.96,12.75,1.56,0.40,37.22,43.32,306.26,8.90,11.44,Wheat,Sowing,Kharif,Yes,84.03,North,Medium,56.29,10.49490
4,Clay,7.76,18.58,0.95,2.52,22.38,86.44,1875.63,10.39,11.26,Cotton,Sowing,Zaid,No,60.86,South,Medium,48.94,41.90145
5,Silt,5.10,20.50,0.37,1.43,33.34,62.51,402.92,7.03,13.55,Rice,Sowing,Zaid,Yes,7.33,East,Medium,55.60,6.63830
6,Sandy,7.44,22.70,0.59,2.95,28.02,58.50,764.95,7.08,7.38,Rice,Flowering,Kharif,No,51.47,West,Medium,63.31,26.16425
7,Sandy,7.68,40.23,0.62,3.30,35.60,79.10,833.33,5.17,3.73,Wheat,Vegetative,Zaid,Yes,118.96,West,Low,39.11,61.80895
8,Clay,5.42,36.73,0.74,1.52,16.59,76.88,2476.03,10.34,1.80,Maize,Flowering,Kharif,Yes,107.25,Central,Low,25.55,85.74245
9,Loamy,6.26,19.58,0.36,1.67,31.14,74.30,1474.55,8.53,17.68,Rice,Vegetative,Rabi,Yes,97.92,Central,Medium,49.93,39.32525


## 02. Séparation entre X et y

Dans cette partie, nous séparons les variables explicatives et la variable cible.

- `X` : les variables utilisées pour prédire
- `y` : la variable cible `Soil_Moisture_J1`

Nous supprimons `Irrigation_Need` car elle correspond à la cible de classification, alors que ce notebook traite un problème de régression.

In [6]:
X = df.drop(["Soil_Moisture_J1", "Irrigation_Need"], axis=1)
y = df["Soil_Moisture_J1"]

print("Shape de X :", X.shape)
print("Shape de y :", y.shape)

Shape de X : (10000, 17)
Shape de y : (10000,)


## 03. Identification des colonnes numériques et catégorielles

Avant le prétraitement, nous devons identifier deux types de colonnes :

- les colonnes numériques,
- les colonnes catégorielles.


In [9]:
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = X.select_dtypes(include=["object", "string"]).columns

print("Colonnes numériques :")
print(numeric_cols)

print("\nNombre de colonnes numériques :", len(numeric_cols))

print("\nColonnes catégorielles :")
print(categorical_cols)

print("\nNombre de colonnes catégorielles :", len(categorical_cols))

Colonnes numériques :
Index(['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity',
       'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours',
       'Wind_Speed_kmh', 'Previous_Irrigation_mm', 'Stress_Index'],
      dtype='str')

Nombre de colonnes numériques : 11

Colonnes catégorielles :
Index(['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season',
       'Mulching_Used', 'Region'],
      dtype='str')

Nombre de colonnes catégorielles : 6


## 04. Split train/test

Nous divisons la dataset en :

- 80% pour l'entraînement,
- 20% pour le test.

Dans un problème de régression, nous n'utilisons pas `stratify`, car la cible est une valeur numérique et non une classe.

In [10]:
from sklearn.model_selection import train_test_split    


In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


print("Shape de X_train :", X_train.shape)
print("Shape de X_test :", X_test.shape)
print("Shape de y_train :", y_train.shape)
print("Shape de y_test :", y_test.shape)

Shape de X_train : (8000, 17)
Shape de X_test : (2000, 17)
Shape de y_train : (8000,)
Shape de y_test : (2000,)


## 05.Standardisation des variables numériques

Dans cette étape, nous allons standardiser les colonnes numériques de la dataset.

La standardisation permet de transformer les variables numériques afin qu'elles aient une moyenne proche de 0 et un écart-type proche de 1.





In [13]:
from sklearn.preprocessing import StandardScaler

In [15]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled = scaler.fit_transform(X_test[numeric_cols])

## 06.Encodage des variables catégorielles

Dans cette étape, nous allons encoder les colonnes catégorielles.

elles contiennent des valeurs textuelles.

Cette méthode transforme chaque catégorie en une nouvelle colonne numérique contenant 0 ou 1.


In [16]:
from  sklearn.preprocessing import OneHotEncoder

In [19]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

X_train_encoded = encoder.fit_transform(X_train[categorical_cols])
X_test_encoded = encoder.transform(X_test[categorical_cols])

## 07.Fusion des données prétraitées

Après la standardisation des variables numériques et l'encodage des variables catégorielles, nous devons regrouper les deux parties dans une seule matrice finale.

Nous combinons donc :

- les colonnes numériques standardisées,
- les colonnes catégorielles encodées.

Cette matrice finale sera utilisée pour entraîner le modèle `RandomForestRegressor`.

In [21]:
X_train_prepared = np.hstack([X_train_scaled, X_train_encoded])
X_test_prepared = np.hstack([X_test_scaled, X_test_encoded])

print("Shape X_train_prepared :", X_train_prepared.shape)
print("Shape X_test_prepared :", X_test_prepared.shape)

Shape X_train_prepared : (8000, 35)
Shape X_test_prepared : (2000, 35)
